In [5]:
"""
Zajel Team - AIC-MTC-4 Dataset Loader
Egypt-Japan University of Science and Technology (E-JUST)

Run this in a Kaggle notebook. Handles:
  - Loading contestant_manifest.json
  - Parsing annotation.txt (x, y, w, h per frame)
  - Iterating video frames with OpenCV
  - Building a PyTorch Dataset for training
  - Generating sample_submission.csv for inference
"""

import os
import json
import csv
import cv2
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch
os.environ["OPENCV_LOG_LEVEL"] = "SILENT" 
# ──────────────────────────────────────────────
# 0. ROOT PATHS  (Kaggle layout)
# ──────────────────────────────────────────────
# Adjust INPUT_ROOT to match your Kaggle dataset mount path
INPUT_ROOT = Path("/kaggle/input/datasets/gamalasran/aic-mtc-4")
MANIFEST_PATH = INPUT_ROOT / "metadata/contestant_manifest.json"
SUBMISSION_TEMPLATE = INPUT_ROOT / "metadata/sample_submission.csv"

# ──────────────────────────────────────────────
# 1. LOAD MANIFEST
# ──────────────────────────────────────────────
def load_manifest(manifest_path: Path) -> dict:
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    print(f"[Manifest] Splits found: {list(manifest.keys())}")
    for split, seqs in manifest.items():
        print(f"  {split:12s}: {len(seqs):4d} sequences")
    return manifest
 
manifest = load_manifest(MANIFEST_PATH)
 
# ──────────────────────────────────────────────
# 2. ANNOTATION PARSER
# ──────────────────────────────────────────────
def load_annotations(annotation_path: Path) -> np.ndarray:
    """
    Returns: np.ndarray of shape (N, 4) — [x, y, w, h] per frame.
    Frames with no visible target are [0, 0, 0, 0].
    Handles both comma-separated AND space-separated annotation formats.
    """
    boxes = []
    with open(annotation_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Support both "x,y,w,h" and "x y w h" formats
            parts = line.replace(",", " ").split()
            boxes.append([float(p) for p in parts[:4]])
    return np.array(boxes, dtype=np.float32)
 
# ──────────────────────────────────────────────
# 3. VIDEO FRAME ITERATOR
# ──────────────────────────────────────────────
def iter_video_frames(video_path: Path):
    """
    Generator — yields (frame_idx, frame_bgr) one frame at a time.
    Memory-efficient: never loads the full video at once.
    Silently skips unreadable files (e.g. moov atom not found).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[WARN] Cannot open video (skipping): {video_path.name}")
        return
 
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        yield frame_idx, frame
        frame_idx += 1
    cap.release()
 
def read_first_frame(video_path: Path) -> np.ndarray:
    """Returns only the first frame (used as the initial template).
    Returns None if the video cannot be read (e.g. moov atom not found)."""
    cap = cv2.VideoCapture(str(video_path))
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print(f"[WARN] Cannot read first frame (skipping): {video_path.name}")
        return None
    return frame  # BGR, HxWx3
 
# ──────────────────────────────────────────────
# 4. SEQUENCE INFO HELPER
# ──────────────────────────────────────────────
def get_sequence_info(manifest: dict, split: str, seq_key: str) -> dict:
    """
    Returns resolved absolute paths for a given sequence.
    seq_key example: 'dataset5/bike1'
    """
    meta = manifest[split][seq_key]
    info = {
        "seq_key":   seq_key,
        "split":     split,
        "n_frames":  meta["n_frames"],
        "fps":       meta["native_fps"],
        "video":     INPUT_ROOT / meta["video_path"],
        "ann":       INPUT_ROOT / meta["annotation_path"] if meta["annotation_path"] else None,
    }
    return info
 
# ──────────────────────────────────────────────
# 5. PYTORCH DATASET — TRAINING
# ──────────────────────────────────────────────
class AerialTrackingDataset(Dataset):
    """
    Returns pairs of (template_patch, search_patch, gt_box) for training.
 
    Strategy:
      - template = first frame crop around GT box (64x64)
      - search   = current frame full crop (128x128)
      - gt_box   = [x, y, w, h] normalized to search region
 
    Skips frames where target is not visible (0,0,0,0).
    """
 
    TEMPLATE_SIZE = 64
    SEARCH_SIZE   = 128
    SEARCH_FACTOR = 2.0   # search region = SEARCH_FACTOR × target area
 
    def __init__(self, manifest: dict, split: str = "train"):
        self.samples = []   # list of (seq_info, frame_idx)
 
        for seq_key, meta in manifest[split].items():
            if meta["annotation_path"] is None:
                continue  # skip sequences without annotations
            info = get_sequence_info(manifest, split, seq_key)
            ann  = load_annotations(info["ann"])
 
            for frame_idx in range(1, len(ann)):   # skip frame 0 (used as template)
                x, y, w, h = ann[frame_idx]
                if w > 0 and h > 0:                # target is visible
                    self.samples.append({
                        "video":     info["video"],
                        "ann":       ann,
                        "frame_idx": frame_idx,
                    })
 
        print(f"[Dataset] {split} — {len(self.samples)} valid frame samples")
 
    def __len__(self):
        return len(self.samples)
 
    def _crop_and_resize(self, frame_bgr, cx, cy, crop_size, out_size):
        """Crop a square region centered at (cx, cy) and resize to out_size."""
        H, W = frame_bgr.shape[:2]
        half = crop_size / 2
        x1 = int(cx - half); y1 = int(cy - half)
        x2 = int(cx + half); y2 = int(cy + half)
 
        # Pad if crop goes outside frame boundaries
        pad_left   = max(0, -x1)
        pad_top    = max(0, -y1)
        pad_right  = max(0, x2 - W)
        pad_bottom = max(0, y2 - H)
        frame_padded = cv2.copyMakeBorder(
            frame_bgr, pad_top, pad_bottom, pad_left, pad_right,
            cv2.BORDER_CONSTANT, value=(114, 114, 114)
        )
        x1 += pad_left; x2 += pad_left
        y1 += pad_top;  y2 += pad_top
        crop = frame_padded[y1:y2, x1:x2]
        crop = cv2.resize(crop, (out_size, out_size))
        return crop
 
    def __getitem__(self, idx):
        sample = self.samples[idx]
        ann    = sample["ann"]
        fid    = sample["frame_idx"]
 
        # ── Template: first frame ──────────────────
        cap = cv2.VideoCapture(str(sample["video"]))
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        _, template_frame = cap.read()
 
        # ── Search: current frame ──────────────────
        cap.set(cv2.CAP_PROP_POS_FRAMES, fid)
        _, search_frame = cap.read()
        cap.release()
 
        # ── Guard: skip bad frames from corrupted videos ──
        if template_frame is None or search_frame is None:
            return self.__getitem__((idx + 1) % len(self))
 
        # ── Template crop ──────────────────────────
        tx, ty, tw, th = ann[0]
        tcx, tcy = tx + tw / 2, ty + th / 2
        t_crop_size = max(tw, th) * self.SEARCH_FACTOR
        template_patch = self._crop_and_resize(
            template_frame, tcx, tcy, t_crop_size, self.TEMPLATE_SIZE
        )
 
        # ── Search crop ────────────────────────────
        sx, sy, sw, sh = ann[fid]
        scx, scy = sx + sw / 2, sy + sh / 2
        s_crop_size = max(sw, sh) * self.SEARCH_FACTOR * 2
        search_patch = self._crop_and_resize(
            search_frame, scx, scy, s_crop_size, self.SEARCH_SIZE
        )
 
        # ── Normalize GT box to [0,1] inside search region ──
        # Convert to search-relative coords
        rx = (scx - s_crop_size / 2)
        ry = (scy - s_crop_size / 2)
        norm_box = np.array([
            (sx - rx) / s_crop_size,
            (sy - ry) / s_crop_size,
            sw        / s_crop_size,
            sh        / s_crop_size,
        ], dtype=np.float32)
 
        # ── To tensor (C, H, W), float32, [0, 1] ──
        template_t = torch.from_numpy(template_patch).permute(2, 0, 1).float() / 255.0
        search_t   = torch.from_numpy(search_patch).permute(2, 0, 1).float() / 255.0
        gt_box_t   = torch.from_numpy(norm_box)
 
        return {
            "template": template_t,   # (3, 64, 64)
            "search":   search_t,     # (3, 128, 128)
            "gt_box":   gt_box_t,     # (4,)  [x, y, w, h] normalized
        }
 
# ──────────────────────────────────────────────
# 6. INFERENCE HELPER — PUBLIC LEADERBOARD
# ──────────────────────────────────────────────
def run_inference_and_save(
    tracker,               # your model/tracker object with .init() and .track()
    manifest: dict,
    submission_template: Path,
    output_csv: Path = Path("/kaggle/working/submission.csv"),
):
    """
    Loops over all public_lb sequences, runs your tracker, writes submission CSV.
 
    tracker interface expected:
        tracker.init(frame_bgr, box_xywh)  → initializes on first frame
        tracker.track(frame_bgr)           → returns (x, y, w, h) prediction
    """
    # Load template row order from sample_submission
    row_order = []
    with open(submission_template, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_order.append(row["id"])
 
    predictions = {}
 
    for seq_key, meta in manifest["public_lb"].items():
        video_path = INPUT_ROOT / meta["video_path"]
        print(f"  Tracking: {seq_key} ({meta['n_frames']} frames)")
 
        for frame_idx, frame_bgr in iter_video_frames(video_path):
            row_id = f"{seq_key}_{frame_idx}"
 
            if frame_idx == 0:
                # First frame: no annotation provided — use full frame center as dummy init
                H, W = frame_bgr.shape[:2]
                init_box = (W // 4, H // 4, W // 2, H // 2)  # replace with GT if provided
                tracker.init(frame_bgr, init_box)
                predictions[row_id] = list(init_box)
            else:
                pred_box = tracker.track(frame_bgr)   # returns (x, y, w, h)
                predictions[row_id] = list(pred_box)
 
    # Write CSV in the exact order of the template
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "x", "y", "w", "h"])
        for row_id in row_order:
            box = predictions.get(row_id, [0, 0, 0, 0])
            writer.writerow([row_id] + [int(v) for v in box])
 
    print(f"\n[✓] Submission saved to: {output_csv}")
    return output_csv
 
# ──────────────────────────────────────────────
# 7. QUICK SANITY CHECK  (run this first!)
# ──────────────────────────────────────────────
if __name__ == "__main__":
 
    # ── Check manifest loads ───────────────────
    manifest = load_manifest(MANIFEST_PATH)
 
    # ── Peek at one training sequence ─────────
    first_train_key = list(manifest["train"].keys())[0]
    info = get_sequence_info(manifest, "train", first_train_key)
    print(f"\nFirst train sequence : {first_train_key}")
    print(f"  Video path         : {info['video']}")
    print(f"  Frames             : {info['n_frames']}  @ {info['fps']} FPS")
 
    ann = load_annotations(info["ann"])
    print(f"  Annotation shape   : {ann.shape}")
    print(f"  First 3 boxes      :\n{ann[:3]}")
 
    # ── Read first frame ───────────────────────
    frame0 = read_first_frame(info["video"])
    print(f"  First frame shape  : {frame0.shape}  (H x W x C)")
 
    # ── Build PyTorch dataset ──────────────────
    dataset = AerialTrackingDataset(manifest, split="train")
    loader  = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)
 
    batch = next(iter(loader))
    print(f"\n[DataLoader] Batch shapes:")
    print(f"  template : {batch['template'].shape}")   # (8, 3, 64, 64)
    print(f"  search   : {batch['search'].shape}")     # (8, 3, 128, 128)
    print(f"  gt_box   : {batch['gt_box'].shape}")     # (8, 4)
    print("\n[✓] Dataset loader is working correctly.")
 

[Manifest] Splits found: ['train', 'public_lb']
  train       :  255 sequences
  public_lb   :   89 sequences
[Manifest] Splits found: ['train', 'public_lb']
  train       :  255 sequences
  public_lb   :   89 sequences

First train sequence : dataset1/Car_video_2
  Video path         : /kaggle/input/datasets/gamalasran/aic-mtc-4/dataset1/Car_video_2/Car_video_2.mp4
  Frames             : 678  @ 60 FPS
  Annotation shape   : (678, 4)
  First 3 boxes      :
[[383. 545. 156.  74.]
 [387. 546. 156.  74.]
 [388. 545. 156.  75.]]
  First frame shape  : (1168, 778, 3)  (H x W x C)
[Dataset] train — 217424 valid frame samples

[DataLoader] Batch shapes:
  template : torch.Size([8, 3, 64, 64])
  search   : torch.Size([8, 3, 128, 128])
  gt_box   : torch.Size([8, 4])

[✓] Dataset loader is working correctly.


In [ ]:
!pip install einops

In [6]:
"""
Zajel Team - 2D-Aware Mamba State Space Backbone
Egypt-Japan University of Science and Technology (E-JUST)

Architecture components:
  1. MambaSSMCore      — Selective SSM with ZOH discretization, O(N) complexity
  2. CrossScanModule   — Scans feature map in 4 directions (H, V, D1, D2)
  3. Mamba2DBlock      — CSM + SSM + residual + layer norm
  4. MambaBackbone     — Stacked Mamba2DBlocks (shared for template & search)
  5. ZajelBackbone     — Parallel dual-stream backbone (template + search)

No external mamba-ssm CUDA kernels required — pure PyTorch.
Compatible with Kaggle's default environment.
"""

import math
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

# ──────────────────────────────────────────────────────────────
# LOGGER SETUP
# ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ZajelBackbone")

# ──────────────────────────────────────────────────────────────
# 0. UTILITY — PARAMETER COUNT
# ──────────────────────────────────────────────────────────────
def count_params(model: nn.Module, label: str = "") -> int:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log.info(f"[Params] {label or model.__class__.__name__}: "
             f"total={total/1e6:.3f}M  trainable={trainable/1e6:.3f}M")
    return total


# ──────────────────────────────────────────────────────────────
# 1. SELECTIVE SSM CORE  (Mamba-style, ZOH discretization)
# ──────────────────────────────────────────────────────────────
class MambaSSMCore(nn.Module):
    """
    Implements the selective SSM recurrence:
        h_t = Ā_t · h_{t-1} + B̄_t · x_t
        y_t = C_t · h_t + D · x_t

    Parameters A, B, C, Δ are INPUT-DEPENDENT (selective),
    making this an S6 block as in the original Mamba paper.

    Discretization: Zero-Order Hold (ZOH)
        Ā = exp(Δ · A)
        B̄ = (Ā − I) · A⁻¹ · B  ≈  Δ · B  (diag approx)

    Complexity: O(N · d_state) per token  →  O(N) overall.
    """

    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model  = d_model
        self.d_state  = d_state
        self.d_inner  = d_model * expand   # expanded inner dimension

        log.debug(f"[MambaSSMCore] d_model={d_model}, d_state={d_state}, "
                  f"d_inner={self.d_inner}, d_conv={d_conv}")

        # ── Input projection: x → z (gate) + x (ssm input) ──
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # ── Depthwise conv for local context before SSM ──
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner,
            bias=True,
        )

        # ── Selective parameter projections (INPUT-DEPENDENT) ──
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)  # → B, C, Δ
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)                 # Δ → full dim

        # ── SSM parameter A: log-initialized, fixed structured ──
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)  # (1, d_state)
        A = A.expand(self.d_inner, -1)                                        # (d_inner, d_state)
        self.A_log = nn.Parameter(torch.log(A))

        # ── Skip connection weight D ──
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # ── Output projection ──
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # ── Activation ──
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, L, d_model)
        Returns:
            y: (B, L, d_model)
        """
        B, L, _ = x.shape
        log.debug(f"  [SSMCore.fwd] input: {x.shape}")

        # ── Project input ──────────────────────────────────
        xz = self.in_proj(x)                           # (B, L, d_inner*2)
        x_in, z = xz.chunk(2, dim=-1)                  # each (B, L, d_inner)

        # ── Local depthwise conv ───────────────────────────
        x_conv = x_in.transpose(1, 2)                  # (B, d_inner, L)
        x_conv = self.conv1d(x_conv)[..., :L]          # causal trim
        x_conv = x_conv.transpose(1, 2)                # (B, L, d_inner)
        x_conv = self.act(x_conv)

        # ── Selective parameters from input ────────────────
        bcd = self.x_proj(x_conv)                      # (B, L, d_state*2 + 1)
        B_sel = bcd[..., :self.d_state]                # (B, L, d_state)
        C_sel = bcd[..., self.d_state:2*self.d_state]  # (B, L, d_state)
        dt    = bcd[..., -1:]                          # (B, L, 1)
        dt    = F.softplus(self.dt_proj(dt))           # (B, L, d_inner)

        # ── ZOH Discretization ─────────────────────────────
        A = -torch.exp(self.A_log)                     # (d_inner, d_state), negative
        # Ā = exp(Δ · A),  B̄ = Δ · B  (diagonal approx)
        dA = torch.exp(dt.unsqueeze(-1) * A)           # (B, L, d_inner, d_state)
        dB = dt.unsqueeze(-1) * B_sel.unsqueeze(2)     # (B, L, d_inner, d_state)

        # ── Sequential scan (O(N) recurrence) ─────────────
        # h: (B, d_inner, d_state)
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            h = dA[:, t] * h + dB[:, t] * x_conv[:, t].unsqueeze(-1)
            # y_t = C_t · h_t
            y_t = (h * C_sel[:, t].unsqueeze(1)).sum(-1)  # (B, d_inner)
            ys.append(y_t)
        y_ssm = torch.stack(ys, dim=1)                 # (B, L, d_inner)

        # ── Skip connection + gate ─────────────────────────
        y_ssm = y_ssm + x_conv * self.D               # D skip
        y_gated = y_ssm * self.act(z)                  # SiLU gate

        # ── Output projection ──────────────────────────────
        out = self.out_proj(y_gated)                   # (B, L, d_model)
        log.debug(f"  [SSMCore.fwd] output: {out.shape}")
        return out


# ──────────────────────────────────────────────────────────────
# 2. CROSS-SCAN MODULE  (4-directional spatial scanning)
# ──────────────────────────────────────────────────────────────
class CrossScanModule(nn.Module):
    """
    Scans a 2D feature map in 4 directions to preserve spatial continuity:
        Direction 0: left  → right  (row-major)
        Direction 1: right → left   (row-major reversed)
        Direction 2: top   → bottom (column-major)
        Direction 3: bottom→ top    (column-major reversed)

    Each direction gets its own SSM scan.
    Outputs are summed and projected back to d_model.
    """

    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.n_dirs  = 4

        log.debug(f"[CrossScanModule] d_model={d_model}, n_dirs=4")

        # One SSM per scan direction
        self.ssm_blocks = nn.ModuleList([
            MambaSSMCore(d_model, d_state=d_state, expand=expand)
            for _ in range(self.n_dirs)
        ])

        # Merge 4 outputs → d_model
        self.merge_proj = nn.Linear(d_model * self.n_dirs, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def _flatten_direction(self, x: torch.Tensor, direction: int) -> torch.Tensor:
        """
        x: (B, H, W, C)
        Returns: (B, H*W, C) scanned in the given direction.
        """
        if direction == 0:   # left → right
            return rearrange(x, 'b h w c -> b (h w) c')
        elif direction == 1: # right → left
            return rearrange(x.flip(2), 'b h w c -> b (h w) c')
        elif direction == 2: # top → bottom
            return rearrange(x.permute(0, 2, 1, 3), 'b h w c -> b (h w) c')
        else:                # bottom → top
            return rearrange(x.flip(1).permute(0, 2, 1, 3), 'b h w c -> b (h w) c')

    def _unflatten_direction(self, y: torch.Tensor, H: int, W: int, direction: int) -> torch.Tensor:
        """
        y: (B, H*W, C)  →  (B, H, W, C) restoring original spatial layout.
        """
        if direction == 0:
            return rearrange(y, 'b (h w) c -> b h w c', h=H, w=W)
        elif direction == 1:
            return rearrange(y, 'b (h w) c -> b h w c', h=H, w=W).flip(2)
        elif direction == 2:
            return rearrange(y, 'b (h w) c -> b w h c', h=W, w=H)
        else:
            return rearrange(y, 'b (h w) c -> b w h c', h=W, w=H).flip(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, C, H, W)  — standard PyTorch image format
        Returns:
            out: (B, C, H, W)
        """
        B, C, H, W = x.shape
        log.debug(f"  [CSM.fwd] input: {x.shape} → scanning {H}×{W}={H*W} tokens in 4 dirs")

        # (B, C, H, W) → (B, H, W, C) for spatial ops
        x_hw = x.permute(0, 2, 3, 1)

        scanned = []
        for d in range(self.n_dirs):
            seq   = self._flatten_direction(x_hw, d)          # (B, H*W, C)
            y_seq = self.ssm_blocks[d](seq)                    # (B, H*W, C)
            y_2d  = self._unflatten_direction(y_seq, H, W, d) # (B, H, W, C)
            scanned.append(y_2d)
            log.debug(f"    [CSM] direction {d} scan complete: {y_2d.shape}")

        # Concatenate all directions and project back
        merged = torch.cat(scanned, dim=-1)                    # (B, H, W, C*4)
        out_hw = self.merge_proj(merged)                       # (B, H, W, C)
        out_hw = self.norm(out_hw)

        out = out_hw.permute(0, 3, 1, 2)                       # (B, C, H, W)
        log.debug(f"  [CSM.fwd] output: {out.shape}")
        return out


# ──────────────────────────────────────────────────────────────
# 3. MAMBA 2D BLOCK  (CSM + FFN + residual connections)
# ──────────────────────────────────────────────────────────────
class Mamba2DBlock(nn.Module):
    """
    Full Mamba2D residual block:
        x → LayerNorm → CrossScanModule → + residual
          → LayerNorm → FFN              → + residual
    """

    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, ffn_expand: int = 4):
        super().__init__()
        log.debug(f"[Mamba2DBlock] d_model={d_model}")

        self.norm1 = nn.LayerNorm(d_model)
        self.csm   = CrossScanModule(d_model, d_state=d_state, expand=expand)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_expand),
            nn.GELU(),
            nn.Linear(d_model * ffn_expand, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args / Returns: (B, C, H, W)
        """
        B, C, H, W = x.shape

        # ── Branch 1: Cross-Scan SSM ───────────────────────
        # Apply LayerNorm in channel dim (treat C as the feature dim)
        x_norm = x.permute(0, 2, 3, 1)             # (B, H, W, C)
        x_norm = self.norm1(x_norm).permute(0, 3, 1, 2)  # back to (B, C, H, W)
        x = x + self.csm(x_norm)                   # residual

        # ── Branch 2: FFN ──────────────────────────────────
        x_flat = x.permute(0, 2, 3, 1)             # (B, H, W, C)
        x_flat = self.norm2(x_flat)
        x_flat = self.ffn(x_flat)
        x = x + x_flat.permute(0, 3, 1, 2)        # residual

        return x


# ──────────────────────────────────────────────────────────────
# 4. PATCH EMBEDDING
# ──────────────────────────────────────────────────────────────
class PatchEmbed(nn.Module):
    """
    Splits image into non-overlapping patches and projects to d_model.
    E.g. 64×64 image with patch_size=8 → 8×8 = 64 tokens.
    """

    def __init__(self, img_size: int, patch_size: int, in_chans: int, d_model: int):
        super().__init__()
        self.img_size   = img_size
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.grid_size  = img_size // patch_size

        self.proj = nn.Conv2d(
            in_chans, d_model,
            kernel_size=patch_size,
            stride=patch_size
        )
        self.norm = nn.LayerNorm(d_model)

        log.debug(f"[PatchEmbed] {img_size}×{img_size} img, "
                  f"patch={patch_size} → {self.grid_size}×{self.grid_size}={self.n_patches} tokens, "
                  f"d_model={d_model}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:  x: (B, 3, H, W)
        Returns: (B, d_model, H//P, W//P)
        """
        x = self.proj(x)                              # (B, d_model, H/P, W/P)
        B, C, H, W = x.shape
        x_norm = x.permute(0, 2, 3, 1)               # (B, H, W, C)
        x_norm = self.norm(x_norm).permute(0, 3, 1, 2)
        log.debug(f"  [PatchEmbed.fwd] {x.shape}")
        return x_norm


# ──────────────────────────────────────────────────────────────
# 5. MAMBA BACKBONE  (stacked blocks)
# ──────────────────────────────────────────────────────────────
class MambaBackbone(nn.Module):
    """
    Full Mamba backbone for one stream (template or search).

    Architecture:
        PatchEmbed → [Mamba2DBlock × depth] → feature map (B, d_model, H/P, W/P)
    """

    def __init__(
        self,
        img_size:   int = 128,
        patch_size: int = 8,
        in_chans:   int = 3,
        d_model:    int = 128,
        depth:      int = 4,
        d_state:    int = 16,
        expand:     int = 2,
    ):
        super().__init__()
        self.img_size  = img_size
        self.patch_size = patch_size
        self.d_model   = d_model
        self.feat_size = img_size // patch_size   # spatial dim of output feature map

        log.info(f"[MambaBackbone] img={img_size}×{img_size}, patch={patch_size}, "
                 f"d_model={d_model}, depth={depth}, d_state={d_state}")

        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, d_model)

        self.blocks = nn.ModuleList([
            Mamba2DBlock(d_model, d_state=d_state, expand=expand)
            for i in range(depth)
        ])

        self.norm = nn.LayerNorm(d_model)

        log.info(f"[MambaBackbone] Output feature map: "
                 f"(B, {d_model}, {self.feat_size}, {self.feat_size})")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:   x: (B, 3, img_size, img_size)
        Returns:   (B, d_model, feat_size, feat_size)
        """
        log.debug(f"[Backbone.fwd] input: {x.shape}")

        x = self.patch_embed(x)       # (B, d_model, H/P, W/P)

        for i, blk in enumerate(self.blocks):
            x_pre = x
            x = blk(x)
            log.debug(f"  [Backbone] block {i+1}/{len(self.blocks)} "
                      f"in={x_pre.shape} out={x.shape}")

        # Final norm (channel-last then back)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = x.permute(0, 3, 1, 2)    # (B, d_model, H/P, W/P)

        log.debug(f"[Backbone.fwd] output: {x.shape}")
        return x


# ──────────────────────────────────────────────────────────────
# 6. DUAL-STREAM ZAJEL BACKBONE
#    Parallel template + search streams (shared weights)
# ──────────────────────────────────────────────────────────────
class ZajelBackbone(nn.Module):
    """
    Dual-stream backbone as described in the Zajel architecture:
      - Template stream:  64×64  → feature map (B, d_model, 8,  8 )
      - Search stream:   128×128 → feature map (B, d_model, 16, 16)

    Both streams share the same Mamba backbone weights,
    reducing parameter count significantly.

    Output features are ready for the Decoupled Correlation head.
    """

    def __init__(
        self,
        d_model:    int = 128,
        depth:      int = 4,
        patch_size: int = 8,
        d_state:    int = 16,
        expand:     int = 2,
    ):
        super().__init__()
        log.info("=" * 60)
        log.info("[ZajelBackbone] Initializing dual-stream Mamba backbone")
        log.info(f"  d_model={d_model}, depth={depth}, patch={patch_size}, "
                 f"d_state={d_state}, expand={expand}")

        # ── Shared backbone (processes both template & search) ──
        # Uses the larger search size; template is upsampled to match
        self.backbone = MambaBackbone(
            img_size=128, patch_size=patch_size,
            in_chans=3, d_model=d_model,
            depth=depth, d_state=d_state, expand=expand,
        )

        # ── Decoupled cross-correlation (θ) ───────────────────
        # Projects template features to a kernel for correlation
        self.template_proj = nn.Conv2d(d_model, d_model, kernel_size=1)
        self.search_proj   = nn.Conv2d(d_model, d_model, kernel_size=1)

        count_params(self, "ZajelBackbone")
        log.info("=" * 60)

    def forward(
        self,
        template: torch.Tensor,   # (B, 3, 64, 64)
        search:   torch.Tensor,   # (B, 3, 128, 128)
    ) -> dict:
        """
        Returns:
            {
              "f_t": (B, d_model, 8,  8),   template features
              "f_s": (B, d_model, 16, 16),  search features
              "fused": (B, d_model, 16, 16) cross-correlated features
            }
        """
        log.debug(f"[ZajelBackbone.fwd] template={template.shape}, search={search.shape}")

        # ── Upsample template to 128×128 to share backbone ──
        template_up = F.interpolate(template, size=(128, 128), mode='bilinear', align_corners=False)
        log.debug(f"  Template upsampled: {template.shape} → {template_up.shape}")

        # ── Parallel feature extraction (shared weights) ──
        f_t_full = self.backbone(template_up)  # (B, d_model, 16, 16)
        f_s      = self.backbone(search)       # (B, d_model, 16, 16)

        # ── Downsample template features back to 8×8 ──
        f_t = F.adaptive_avg_pool2d(f_t_full, (8, 8))  # (B, d_model, 8, 8)
        log.debug(f"  f_t (template features): {f_t.shape}")
        log.debug(f"  f_s (search features):   {f_s.shape}")

        # ── Decoupled cross-correlation ──────────────────────
        # Project both to correlation space
        f_t_proj = self.template_proj(f_t)    # (B, d_model, 8, 8)
        f_s_proj = self.search_proj(f_s)      # (B, d_model, 16, 16)

        # Upsample template kernel and element-wise multiply
        f_t_up = F.interpolate(f_t_proj, size=f_s_proj.shape[-2:],
                               mode='bilinear', align_corners=False)
        fused = f_s_proj * f_t_up             # (B, d_model, 16, 16)
        log.debug(f"  fused (cross-corr):      {fused.shape}")

        return {"f_t": f_t, "f_s": f_s, "fused": fused}


# ──────────────────────────────────────────────────────────────
# 7. SANITY CHECK
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import torch

    log.info("Running Zajel Mamba Backbone sanity check...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {device}")

    # ── Build model ───────────────────────────────────────────
    model = ZajelBackbone(
        d_model=128,   # feature dimension
        depth=4,       # number of Mamba2D blocks
        patch_size=8,  # 64/8=8 template tokens, 128/8=16 search tokens
        d_state=16,    # SSM state size
        expand=2,      # SSM inner expansion
    ).to(device)

    # ── Dummy batch ───────────────────────────────────────────
    B = 4
    template = torch.randn(B, 3, 64, 64).to(device)
    search   = torch.randn(B, 3, 128, 128).to(device)

    log.info(f"Input  — template: {template.shape}, search: {search.shape}")

    # ── Forward pass ──────────────────────────────────────────
    with torch.no_grad():
        out = model(template, search)

    log.info("── Output Feature Maps ─────────────────────────────")
    for k, v in out.items():
        log.info(f"  {k:8s}: {v.shape}")

    # ── Parameter budget check ────────────────────────────────
    total = sum(p.numel() for p in model.parameters())
    log.info(f"\n── Parameter Budget ────────────────────────────────")
    log.info(f"  Total params : {total/1e6:.3f}M  (limit: 10M)")
    log.info(f"  Budget used  : {total/10e6*100:.1f}%")

    assert out["f_t"].shape   == (B, 128, 8,  8),  f"f_t shape mismatch: {out['f_t'].shape}"
    assert out["f_s"].shape   == (B, 128, 16, 16), f"f_s shape mismatch: {out['f_s'].shape}"
    assert out["fused"].shape == (B, 128, 16, 16), f"fused shape mismatch: {out['fused'].shape}"

    log.info("\n[✓] Backbone sanity check passed.")
    log.info("    Ready for Step 3: Classification + NWD Regression Heads.")

09:00:09 | INFO     | ZajelBackbone | Running Zajel Mamba Backbone sanity check...
09:00:09 | INFO     | ZajelBackbone | Device: cuda
09:00:09 | INFO     | ZajelBackbone | ============================================================
09:00:09 | INFO     | ZajelBackbone | [ZajelBackbone] Initializing dual-stream Mamba backbone
09:00:09 | INFO     | ZajelBackbone |   d_model=128, depth=4, patch=8, d_state=16, expand=2
09:00:09 | INFO     | ZajelBackbone | [MambaBackbone] img=128×128, patch=8, d_model=128, depth=4, d_state=16
09:00:09 | DEBUG    | ZajelBackbone | [PatchEmbed] 128×128 img, patch=8 → 16×16=256 tokens, d_model=128
09:00:09 | DEBUG    | ZajelBackbone | [Mamba2DBlock] d_model=128
09:00:09 | DEBUG    | ZajelBackbone | [CrossScanModule] d_model=128, n_dirs=4
09:00:09 | DEBUG    | ZajelBackbone | [MambaSSMCore] d_model=128, d_state=16, d_inner=256, d_conv=4
09:00:09 | DEBUG    | ZajelBackbone | [MambaSSMCore] d_model=128, d_state=16, d_inner=256, d_conv=4
09:00:09 | DEBUG    | Zaj

In [8]:
"""
Zajel Team - Classification & NWD Regression Heads
Egypt-Japan University of Science and Technology (E-JUST)

Components:
  1. ClassificationHead   — Confidence score map (B, 1, H, W)
  2. NWDRegressionHead    — Bounding box as 2D Gaussian, predicts (cx, cy, w, h)
  3. NWDLoss              — Normalized Wasserstein Distance loss
  4. ZajelHead            — Combines both heads on fused features
  5. Sanity check
"""

import math
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

log = logging.getLogger("ZajelHeads")

# ──────────────────────────────────────────────────────────────
# 1. CLASSIFICATION HEAD
#    Input:  fused (B, d_model, H, W)
#    Output: score map (B, 1, H, W)  — confidence per search cell
# ──────────────────────────────────────────────────────────────
class ClassificationHead(nn.Module):
    """
    Lightweight conv head that produces a per-cell confidence score.

    Each spatial cell (i,j) in the score map answers:
    "Is the target centered near this location?"

    Architecture: Conv → BN → ReLU → Conv → Sigmoid
    """

    def __init__(self, d_model: int, hidden: int = 64):
        super().__init__()
        log.debug(f"[ClsHead] d_model={d_model}, hidden={hidden}")

        self.head = nn.Sequential(
            nn.Conv2d(d_model, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 1, kernel_size=1),   # → (B, 1, H, W)
        )

        self._init_weights()

    def _init_weights(self):
        # Bias init: prior probability ~0.01 to avoid early over-confidence
        prior = 0.01
        bias_val = -math.log((1 - prior) / prior)
        nn.init.constant_(self.head[-1].bias, bias_val)
        log.debug(f"  [ClsHead] bias initialized to {bias_val:.4f} (prior={prior})")

    def forward(self, fused: torch.Tensor) -> torch.Tensor:
        """
        Args:   fused: (B, d_model, H, W)
        Returns: score: (B, 1, H, W)  — raw logits (apply sigmoid for prob)
        """
        score = self.head(fused)
        log.debug(f"  [ClsHead.fwd] {fused.shape} → {score.shape}")
        return score


# ──────────────────────────────────────────────────────────────
# 2. NWD REGRESSION HEAD
#    Input:  fused (B, d_model, H, W)
#    Output: box_map (B, 4, H, W)  — (cx, cy, w, h) offsets per cell
# ──────────────────────────────────────────────────────────────
class NWDRegressionHead(nn.Module):
    """
    Predicts bounding box parameters as offsets from each search grid cell.

    Output per cell: (Δcx, Δcy, w, h) — all normalized to [0,1] relative
    to the search region size.

    The final box is decoded as:
        cx = (cell_x + sigmoid(Δcx)) / grid_W
        cy = (cell_y + sigmoid(Δcy)) / grid_H
        w  = sigmoid(w_pred)
        h  = sigmoid(h_pred)

    Architecture: Conv → BN → ReLU → Conv → Sigmoid
    """

    def __init__(self, d_model: int, hidden: int = 64):
        super().__init__()
        log.debug(f"[RegHead] d_model={d_model}, hidden={hidden}")

        self.head = nn.Sequential(
            nn.Conv2d(d_model, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 4, kernel_size=1),   # → (B, 4, H, W)
            nn.Sigmoid(),                           # clamp to [0, 1]
        )

    def forward(self, fused: torch.Tensor) -> torch.Tensor:
        """
        Args:   fused: (B, d_model, H, W)
        Returns: box_map: (B, 4, H, W)  values in [0,1]
        """
        box_map = self.head(fused)
        log.debug(f"  [RegHead.fwd] {fused.shape} → {box_map.shape}")
        return box_map

    @staticmethod
    def decode(box_map: torch.Tensor) -> torch.Tensor:
        """
        Decode box_map to a single (cx, cy, w, h) prediction by picking
        the cell with highest classification score (called externally).

        Args:
            box_map: (B, 4, H, W)
        Returns:
            boxes: (B, 4)  — cx, cy, w, h each in [0, 1]
        """
        B, _, H, W = box_map.shape

        # Build grid of cell centers
        grid_y, grid_x = torch.meshgrid(
            torch.arange(H, device=box_map.device, dtype=box_map.dtype),
            torch.arange(W, device=box_map.device, dtype=box_map.dtype),
            indexing='ij'
        )

        # box_map[:, 0:2] = offset within cell (already sigmoid → [0,1])
        # Absolute position = (cell + offset) / grid_size
        cx = (grid_x.unsqueeze(0) + box_map[:, 0]) / W   # (B, H, W)
        cy = (grid_y.unsqueeze(0) + box_map[:, 1]) / H   # (B, H, W)
        w  = box_map[:, 2]                                 # (B, H, W)
        h  = box_map[:, 3]                                 # (B, H, W)

        return torch.stack([cx, cy, w, h], dim=1)  # (B, 4, H, W) decoded


# ──────────────────────────────────────────────────────────────
# 3. NWD LOSS
#    Normalized Wasserstein Distance between predicted and GT boxes
#    modeled as 2D Gaussian distributions.
# ──────────────────────────────────────────────────────────────
class NWDLoss(nn.Module):
    """
    Normalized Wasserstein Distance (NWD) loss for bounding box regression.

    Instead of IoU (which gives zero gradient for non-overlapping boxes),
    NWD models each box as a 2D Gaussian:
        N(μ, Σ)  where  μ = (cx, cy),  Σ = diag(w/2, h/2)²

    The 2nd-order Wasserstein distance between two Gaussians is:
        W²(p, q) = ||μ_p − μ_q||² + ||√Σ_p − √Σ_q||²_F
                 = (cx_p−cx_q)² + (cy_p−cy_q)² + (w_p/2−w_q/2)² + (h_p/2−h_q/2)²

    Normalized to [0,1]:
        NWD = exp(−W₂ / C)    where C is a normalization constant

    Loss = 1 − NWD  (so 0 = perfect match, 1 = worst case)

    Key advantage: smooth gradients even with ZERO pixel overlap,
    critical for microscopic aerial targets.
    """

    def __init__(self, C: float = 0.5):
        """
        Args:
            C: normalization constant — controls sensitivity.
               IMPORTANT: use C=0.5 for normalized [0,1] coords, C=12.8 for pixel coords.
        """
        super().__init__()
        self.C = C
        log.debug(f"[NWDLoss] normalization constant C={C}")

    def forward(
        self,
        pred: torch.Tensor,   # (B, 4)  — cx, cy, w, h normalized [0,1]
        target: torch.Tensor, # (B, 4)  — cx, cy, w, h normalized [0,1]
        weight: torch.Tensor = None,  # (B,) optional per-sample weight
    ) -> torch.Tensor:
        """
        Returns: scalar NWD loss.
        """
        # Gaussian means
        cx_p, cy_p, w_p, h_p = pred[:, 0],   pred[:, 1],   pred[:, 2],   pred[:, 3]
        cx_t, cy_t, w_t, h_t = target[:, 0], target[:, 1], target[:, 2], target[:, 3]

        # W² distance between Gaussian parameters
        # σ = side/2 for each axis (diagonal covariance)
        w2 = (
            (cx_p - cx_t) ** 2 +
            (cy_p - cy_t) ** 2 +
            (w_p / 2 - w_t / 2) ** 2 +
            (h_p / 2 - h_t / 2) ** 2
        )  # (B,)

        nwd = torch.exp(-w2.sqrt() / self.C)   # (B,)  ∈ [0, 1]
        loss = 1.0 - nwd                        # (B,)

        log.debug(f"  [NWDLoss] W²={w2.mean():.4f}  NWD={nwd.mean():.4f}  loss={loss.mean():.4f}")

        if weight is not None:
            loss = loss * weight

        return loss.mean()


# ──────────────────────────────────────────────────────────────
# 4. COMBINED ZAJEL HEAD
# ──────────────────────────────────────────────────────────────
class ZajelHead(nn.Module):
    """
    Combines Classification + NWD Regression heads on the fused feature map.

    Forward returns:
        score_map: (B, 1,  H, W)  — confidence logits per cell
        box_map:   (B, 4,  H, W)  — (cx, cy, w, h) per cell in [0,1]
        pred_box:  (B, 4)         — final predicted box (from argmax cell)

    Loss (training):
        cls_loss  = FocalLoss(score_map, cls_target)
        reg_loss  = NWDLoss(pred_box, gt_box)
        total     = cls_loss + λ * reg_loss
    """

    def __init__(self, d_model: int, hidden: int = 64, nwd_C: float = 0.5):
        super().__init__()
        log.info(f"[ZajelHead] d_model={d_model}, hidden={hidden}, nwd_C={nwd_C}")

        self.cls_head = ClassificationHead(d_model, hidden)
        self.reg_head = NWDRegressionHead(d_model, hidden)
        self.nwd_loss = NWDLoss(C=nwd_C)

        # Classification loss: Focal Loss (handles class imbalance)
        # background cells >> foreground cells in each search map
        self.focal_alpha = 0.25
        self.focal_gamma = 2.0

        total = sum(p.numel() for p in self.parameters())
        log.info(f"[ZajelHead] params: {total/1e6:.3f}M")

    def focal_loss(self, pred_logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Sigmoid Focal Loss.
        Args:
            pred_logits: (B, 1, H, W) — raw (pre-sigmoid) scores
            targets:     (B, 1, H, W) — 0/1 Gaussian label map
        """
        p   = torch.sigmoid(pred_logits)
        ce  = F.binary_cross_entropy_with_logits(pred_logits, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        fl  = ce * ((1 - p_t) ** self.focal_gamma)

        alpha_t = self.focal_alpha * targets + (1 - self.focal_alpha) * (1 - targets)
        fl = alpha_t * fl

        loss = fl.mean()
        log.debug(f"  [FocalLoss] loss={loss.item():.4f}")
        return loss

    def forward(
        self,
        fused: torch.Tensor,           # (B, d_model, H, W)
        gt_box: torch.Tensor = None,   # (B, 4) cx,cy,w,h normalized — only in training
    ) -> dict:
        """
        Returns dict with:
            score_map, box_map, pred_box  — always
            cls_loss, reg_loss, loss      — only if gt_box is provided
        """
        B, _, H, W = fused.shape
        log.debug(f"[ZajelHead.fwd] fused={fused.shape}, training={gt_box is not None}")

        # ── Forward through heads ─────────────────────────
        score_map = self.cls_head(fused)         # (B, 1, H, W)
        box_map   = self.reg_head(fused)         # (B, 4, H, W)

        # ── Decode best box (argmax of score map) ─────────
        score_flat = score_map.view(B, -1)                          # (B, H*W)
        best_idx   = score_flat.argmax(dim=1)                       # (B,)
        best_row   = best_idx // W                                  # (B,)
        best_col   = best_idx %  W                                  # (B,)

        # Gather decoded box at argmax cell
        decoded = NWDRegressionHead.decode(box_map)                 # (B, 4, H, W)
        pred_box = decoded[
            torch.arange(B, device=fused.device),
            :,
            best_row,
            best_col,
        ]                                                           # (B, 4)

        log.debug(f"  [ZajelHead] score_map={score_map.shape}, "
                  f"box_map={box_map.shape}, pred_box={pred_box.shape}")
        log.debug(f"  [ZajelHead] sample pred_box[0]: "
                  f"cx={pred_box[0,0]:.3f} cy={pred_box[0,1]:.3f} "
                  f"w={pred_box[0,2]:.3f} h={pred_box[0,3]:.3f}")

        out = {
            "score_map": score_map,      # (B, 1, H, W)
            "box_map":   box_map,        # (B, 4, H, W)
            "pred_box":  pred_box,       # (B, 4)
        }

        # ── Losses (training only) ────────────────────────
        if gt_box is not None:
            # Build Gaussian label map centered on GT cell
            gt_label_map = self._make_label_map(gt_box, H, W, fused.device)

            cls_loss = self.focal_loss(score_map, gt_label_map)
            reg_loss = self.nwd_loss(pred_box, gt_box)
            total_loss = cls_loss + 2.0 * reg_loss

            log.debug(f"  [ZajelHead] cls_loss={cls_loss.item():.4f}, "
                      f"reg_loss={reg_loss.item():.4f}, "
                      f"total={total_loss.item():.4f}")

            out.update({
                "cls_loss":  cls_loss,
                "reg_loss":  reg_loss,
                "loss":      total_loss,
                "label_map": gt_label_map,
            })

        return out

    def _make_label_map(
        self,
        gt_box: torch.Tensor,  # (B, 4) cx, cy, w, h in [0,1]
        H: int, W: int,
        device: torch.device,
        sigma: float = 0.1,    # Gaussian spread (relative to grid)
    ) -> torch.Tensor:
        """
        Creates a soft Gaussian label map where cells near the GT center
        get higher target values, and background cells get 0.

        This is far better than a hard 0/1 map for microscopic targets.
        """
        B = gt_box.shape[0]

        # GT center in grid coordinates
        cx_g = gt_box[:, 0] * W   # (B,)
        cy_g = gt_box[:, 1] * H   # (B,)

        # Grid of cell centers
        grid_y, grid_x = torch.meshgrid(
            torch.arange(H, device=device, dtype=torch.float32),
            torch.arange(W, device=device, dtype=torch.float32),
            indexing='ij'
        )   # (H, W) each

        # Broadcast: (B, H, W)
        dx = grid_x.unsqueeze(0) - cx_g.view(B, 1, 1)
        dy = grid_y.unsqueeze(0) - cy_g.view(B, 1, 1)

        sigma_px = sigma * max(H, W)
        label = torch.exp(-(dx**2 + dy**2) / (2 * sigma_px**2))  # (B, H, W)
        label = label.unsqueeze(1)                                  # (B, 1, H, W)

        log.debug(f"  [LabelMap] GT cells: cx_g={cx_g[0]:.2f}, cy_g={cy_g[0]:.2f}, "
                  f"peak_val={label[0].max():.3f}")
        return label


# ──────────────────────────────────────────────────────────────
# 5. SANITY CHECK
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    sys.path.append("/kaggle/working")  # adjust if needed

    logging.basicConfig(
        level=logging.DEBUG,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {device}")

    # ── Simulate backbone output ──────────────────────────
    B, d_model, H, W = 4, 128, 16, 16
    fused = torch.randn(B, d_model, H, W).to(device)
    gt_box = torch.tensor([
        [0.5, 0.5, 0.15, 0.12],
        [0.3, 0.7, 0.10, 0.08],
        [0.6, 0.4, 0.20, 0.18],
        [0.2, 0.2, 0.12, 0.10],
    ]).to(device)   # (B, 4) — cx, cy, w, h

    # ── Build head ────────────────────────────────────────
    head = ZajelHead(d_model=d_model, hidden=64, nwd_C=0.5).to(device)

    # ── Inference mode (no GT) ────────────────────────────
    log.info("\n── Inference mode (no gt_box) ──────────────────────")
    with torch.no_grad():
        out = head(fused)
    log.info(f"  score_map : {out['score_map'].shape}")
    log.info(f"  box_map   : {out['box_map'].shape}")
    log.info(f"  pred_box  : {out['pred_box'].shape}")
    log.info(f"  pred_box[0]: {out['pred_box'][0].tolist()}")

    # ── Training mode (with GT) ───────────────────────────
    log.info("\n── Training mode (with gt_box) ─────────────────────")
    out_train = head(fused, gt_box=gt_box)
    log.info(f"  cls_loss  : {out_train['cls_loss'].item():.4f}")
    log.info(f"  reg_loss  : {out_train['reg_loss'].item():.4f}")
    log.info(f"  total loss: {out_train['loss'].item():.4f}")

    # ── Backward pass (gradient check) ───────────────────
    log.info("\n── Backward pass ───────────────────────────────────")
    out_train["loss"].backward()
    grad_norms = {
        name: p.grad.norm().item()
        for name, p in head.named_parameters()
        if p.grad is not None
    }
    log.info(f"  Layers with gradients : {len(grad_norms)}")
    log.info(f"  Max grad norm         : {max(grad_norms.values()):.4f}")
    log.info(f"  Min grad norm         : {min(grad_norms.values()):.6f}")

    # ── NWD loss unit test ────────────────────────────────
    log.info("\n── NWD Loss unit tests ─────────────────────────────")
    nwd = NWDLoss(C=0.5)

    perfect = torch.tensor([[0.5, 0.5, 0.2, 0.2]])
    loss_perfect = nwd(perfect, perfect)
    log.info(f"  Perfect match loss    : {loss_perfect.item():.6f}  (expected ≈ 0.0)")

    far_off = torch.tensor([[0.0, 0.0, 0.1, 0.1]])
    gt_far  = torch.tensor([[1.0, 1.0, 0.5, 0.5]])
    loss_far = nwd(far_off, gt_far)
    log.info(f"  Far-off pred loss     : {loss_far.item():.6f}  (expected > 0.5)")

    # ── Parameter summary ────────────────────────────────
    log.info("\n── Parameter Budget ────────────────────────────────")
    backbone_params = 2_657_000   # from Step 2
    head_params = sum(p.numel() for p in head.parameters())
    total = backbone_params + head_params
    log.info(f"  Backbone params : {backbone_params/1e6:.3f}M")
    log.info(f"  Head params     : {head_params/1e6:.3f}M")
    log.info(f"  Combined total  : {total/1e6:.3f}M  (limit: 10M)")
    log.info(f"  Budget used     : {total/10e6*100:.1f}%")

    assert loss_perfect.item() < 0.01, "Perfect match should have ~0 loss"
    assert loss_far.item() > 0.5,      "Far-off pred should have high loss"
    assert out["pred_box"].shape == (B, 4), f"pred_box shape error: {out['pred_box'].shape}"

    log.info("\n[✓] Heads sanity check passed.")
    log.info("    Ready for Step 4: Triple-Gated Dynamic Template Updater.")

09:05:39 | INFO     | ZajelHeads | Device: cuda
09:05:39 | INFO     | ZajelHeads | [ZajelHead] d_model=128, hidden=64, nwd_C=0.5
09:05:39 | DEBUG    | ZajelHeads | [ClsHead] d_model=128, hidden=64
09:05:39 | DEBUG    | ZajelHeads |   [ClsHead] bias initialized to -4.5951 (prior=0.01)
09:05:39 | DEBUG    | ZajelHeads | [RegHead] d_model=128, hidden=64
09:05:39 | DEBUG    | ZajelHeads | [NWDLoss] normalization constant C=0.5
09:05:39 | INFO     | ZajelHeads | [ZajelHead] params: 0.222M
09:05:39 | INFO     | ZajelHeads | 
── Inference mode (no gt_box) ──────────────────────
09:05:39 | DEBUG    | ZajelHeads | [ZajelHead.fwd] fused=torch.Size([4, 128, 16, 16]), training=False
09:05:39 | DEBUG    | ZajelHeads |   [ClsHead.fwd] torch.Size([4, 128, 16, 16]) → torch.Size([4, 1, 16, 16])
09:05:39 | DEBUG    | ZajelHeads |   [RegHead.fwd] torch.Size([4, 128, 16, 16]) → torch.Size([4, 4, 16, 16])
09:05:39 | DEBUG    | ZajelHeads |   [ZajelHead] score_map=torch.Size([4, 1, 16, 16]), box_map=torch.S

In [10]:
"""
Zajel Team - Triple-Gated Dynamic Template Updater
Egypt-Japan University of Science and Technology (E-JUST)

Implements online-only, drift-free template management:

  Gate 1 — Peak Verification       : cross-correlation response must have a strong peak
  Gate 2 — Unimodal Consistency    : Peak-to-Sidelobe Ratio (PSR) ensures single-mode response
  Gate 3 — Spatial Velocity Check  : predicted displacement must be physically plausible

  Update rule (EMA):
      T_new = α · T_current + (1 − α) · T_first_frame

  All three gates must pass simultaneously for an update to occur.
  On failure the template is frozen, preventing background drift.
"""

import logging
import torch
import torch.nn as nn
import torch.nn.functional as F

log = logging.getLogger("ZajelUpdater")


# ──────────────────────────────────────────────────────────────
# HELPER — 2D CROSS-CORRELATION RESPONSE MAP
# ──────────────────────────────────────────────────────────────
def cross_correlation_response(
    template_feat: torch.Tensor,   # (1, C, Ht, Wt)
    search_feat:   torch.Tensor,   # (1, C, Hs, Ws)
) -> torch.Tensor:
    """
    Computes a 2D response map by sliding template_feat over search_feat.
    Uses depthwise cross-correlation (each channel correlated independently,
    then summed) — lightweight and differentiable.

    Returns: response (1, 1, Hs-Ht+1, Ws-Wt+1)  — higher = better match
    """
    C, Ht, Wt = template_feat.shape[1], template_feat.shape[2], template_feat.shape[3]

    # Reshape template as conv kernel: (C, 1, Ht, Wt) × C groups
    kernel = template_feat.view(C, 1, Ht, Wt)

    # Depthwise conv: output is (1, C, Hs-Ht+1, Ws-Wt+1)
    response_per_ch = F.conv2d(
        search_feat,         # (1, C, Hs, Ws)
        kernel,              # (C, 1, Ht, Wt)
        groups=C,
    )

    # Sum over channels → (1, 1, H_out, W_out)
    response = response_per_ch.sum(dim=1, keepdim=True)
    log.debug(f"  [XCorr] template={template_feat.shape}, search={search_feat.shape} "
              f"→ response={response.shape}")
    return response


# ──────────────────────────────────────────────────────────────
# GATE 1 — PEAK VERIFICATION
# ──────────────────────────────────────────────────────────────
def gate_peak_verification(
    response: torch.Tensor,   # (1, 1, H, W)
    threshold: float = 0.5,   # minimum normalized peak amplitude
) -> tuple[bool, float]:
    """
    Checks that the response map has a sufficiently strong peak.

    Normalized peak = (peak - mean) / (std + ε)

    Returns: (passed: bool, norm_peak: float)
    """
    r = response.squeeze()                           # (H, W)
    peak_val = r.max().item()
    mean_val = r.mean().item()
    std_val  = r.std().item()
    norm_peak = (peak_val - mean_val) / (std_val + 1e-6)

    passed = norm_peak >= threshold
    log.debug(f"  [Gate1-Peak] norm_peak={norm_peak:.4f}, threshold={threshold:.2f} "
              f"→ {'✓ PASS' if passed else '✗ FAIL'}")
    return passed, norm_peak


# ──────────────────────────────────────────────────────────────
# GATE 2 — UNIMODAL CONSISTENCY  (Peak-to-Sidelobe Ratio)
# ──────────────────────────────────────────────────────────────
def gate_psr(
    response: torch.Tensor,   # (1, 1, H, W)
    win_size: int = 7,        # exclusion window around peak
    threshold: float = 3.0,   # minimum PSR value
) -> tuple[bool, float]:
    """
    Peak-to-Sidelobe Ratio (PSR):
        PSR = (peak - sidelobe_mean) / (sidelobe_std + ε)

    Sidelobe = all values OUTSIDE the win_size window around the peak.
    High PSR → single dominant peak → confident, unimodal response.
    Low PSR  → multiple peaks or flat response → ambiguous, DON'T update.

    Returns: (passed: bool, psr: float)
    """
    r = response.squeeze()   # (H, W)
    H, W = r.shape

    # Find peak location
    flat_idx  = r.argmax()
    peak_row  = (flat_idx // W).item()
    peak_col  = (flat_idx %  W).item()
    peak_val  = r[peak_row, peak_col].item()

    # Build exclusion mask around peak
    mask = torch.ones_like(r, dtype=torch.bool)
    half = win_size // 2
    r1 = max(0, peak_row - half);  r2 = min(H, peak_row + half + 1)
    c1 = max(0, peak_col - half);  c2 = min(W, peak_col + half + 1)
    mask[r1:r2, c1:c2] = False   # exclude peak window

    sidelobe = r[mask]
    if sidelobe.numel() == 0:
        log.debug("  [Gate2-PSR] no sidelobe region → FAIL")
        return False, 0.0

    sl_mean = sidelobe.mean().item()
    sl_std  = sidelobe.std().item()
    psr     = (peak_val - sl_mean) / (sl_std + 1e-6)

    passed = psr >= threshold
    log.debug(f"  [Gate2-PSR] peak={peak_val:.4f}, sl_mean={sl_mean:.4f}, "
              f"sl_std={sl_std:.4f}, PSR={psr:.4f}, threshold={threshold:.2f} "
              f"→ {'✓ PASS' if passed else '✗ FAIL'}")
    return passed, psr


# ──────────────────────────────────────────────────────────────
# GATE 3 — SPATIAL VELOCITY CHECK
# ──────────────────────────────────────────────────────────────
def gate_velocity(
    prev_box:     torch.Tensor,   # (4,)  cx, cy, w, h  [normalized]
    pred_box:     torch.Tensor,   # (4,)  cx, cy, w, h  [normalized]
    max_disp:     float = 0.25,   # max allowed displacement per frame
    max_scale:    float = 0.50,   # max allowed scale change ratio
) -> tuple[bool, float, float]:
    """
    Checks that the predicted box has not jumped unrealistically far
    from the previous box, given UAV motion constraints.

    Two sub-checks:
        1. Displacement: Euclidean distance between centers ≤ max_disp
        2. Scale change: |log(w_pred/w_prev)| ≤ log(1+max_scale)

    Returns: (passed: bool, displacement: float, scale_change: float)
    """
    cx_pr, cy_pr, w_pr, h_pr = prev_box[0].item(), prev_box[1].item(), prev_box[2].item(), prev_box[3].item()
    cx_pd, cy_pd, w_pd, h_pd = pred_box[0].item(), pred_box[1].item(), pred_box[2].item(), pred_box[3].item()

    # Sub-check 1: Center displacement
    disp = ((cx_pd - cx_pr)**2 + (cy_pd - cy_pr)**2) ** 0.5

    # Sub-check 2: Scale change (log ratio is symmetric)
    scale_w = abs(torch.log(torch.tensor(w_pd / (w_pr + 1e-6))).item())
    scale_h = abs(torch.log(torch.tensor(h_pd / (h_pr + 1e-6))).item())
    scale_change = max(scale_w, scale_h)

    import math
    max_scale_log = math.log(1 + max_scale)
    disp_ok  = disp         <= max_disp
    scale_ok = scale_change <= max_scale_log
    passed   = disp_ok and scale_ok

    log.debug(f"  [Gate3-Vel] disp={disp:.4f}/{max_disp:.2f} {'✓' if disp_ok else '✗'}  "
              f"scale={scale_change:.4f}/{max_scale_log:.4f} {'✓' if scale_ok else '✗'} "
              f"→ {'✓ PASS' if passed else '✗ FAIL'}")
    return passed, disp, scale_change


# ──────────────────────────────────────────────────────────────
# TRIPLE-GATED DYNAMIC TEMPLATE UPDATER
# ──────────────────────────────────────────────────────────────
class TripleGatedUpdater:
    """
    Manages the template feature across frames using three safety gates.

    State kept per sequence:
        - template_feat  : current working template feature (updated via EMA)
        - first_feat     : pristine first-frame feature (NEVER updated)
        - prev_box       : last confirmed bounding box
        - update_count   : total number of accepted template updates
        - reject_count   : total number of rejected updates (all gates logged)

    Usage:
        updater = TripleGatedUpdater(alpha=0.02)
        updater.init(first_frame_feat, init_box)

        for each frame:
            pred_box, fused = model(template, search)
            template_feat   = updater.step(search_feat, pred_box)
            # use template_feat for next frame
    """

    def __init__(
        self,
        alpha:          float = 0.02,   # EMA weight for new frame (small = conservative)
        peak_thresh:    float = 0.5,    # Gate 1: minimum normalized peak
        psr_thresh:     float = 3.0,    # Gate 2: minimum PSR
        max_disp:       float = 0.25,   # Gate 3: max center displacement
        max_scale:      float = 0.50,   # Gate 3: max scale change ratio
        psr_win:        int   = 7,      # Gate 2: exclusion window size
    ):
        self.alpha       = alpha
        self.peak_thresh = peak_thresh
        self.psr_thresh  = psr_thresh
        self.max_disp    = max_disp
        self.max_scale   = max_scale
        self.psr_win     = psr_win

        # State (initialized in .init())
        self.template_feat  = None
        self.first_feat     = None
        self.prev_box       = None
        self.update_count   = 0
        self.reject_count   = 0
        self.frame_idx      = 0

        # Per-frame gate result history for debugging
        self.gate_log = []   # list of dicts

        log.info(f"[TripleGatedUpdater] α={alpha}, "
                 f"peak_thresh={peak_thresh}, psr_thresh={psr_thresh}, "
                 f"max_disp={max_disp}, max_scale={max_scale}")

    def init(self, first_feat: torch.Tensor, init_box: torch.Tensor):
        """
        Initialize with the first-frame feature and bounding box.

        Args:
            first_feat: (1, C, H, W) — backbone features of frame 0
            init_box:   (4,)         — cx, cy, w, h normalized [0,1]
        """
        self.first_feat    = first_feat.clone()
        self.template_feat = first_feat.clone()
        self.prev_box      = init_box.clone()
        self.update_count  = 0
        self.reject_count  = 0
        self.frame_idx     = 0
        self.gate_log      = []
        log.info(f"[Updater.init] template_feat={first_feat.shape}, "
                 f"init_box={init_box.tolist()}")

    @torch.no_grad()
    def step(
        self,
        search_feat: torch.Tensor,   # (1, C, Hs, Ws) — current frame search features
        pred_box:    torch.Tensor,   # (4,)            — cx, cy, w, h normalized
    ) -> torch.Tensor:
        """
        Evaluates all three gates and conditionally updates the template.

        Returns: current template_feat (updated or frozen)
        """
        self.frame_idx += 1
        log.debug(f"\n[Updater] Frame {self.frame_idx} ─────────────────────")

        # ── Compute cross-correlation response ────────────
        response = cross_correlation_response(self.template_feat, search_feat)

        # ── Gate 1: Peak Verification ──────────────────────
        g1_pass, norm_peak = gate_peak_verification(response, self.peak_thresh)

        # ── Gate 2: PSR Unimodal Consistency ──────────────
        g2_pass, psr = gate_psr(response, self.psr_win, self.psr_thresh)

        # ── Gate 3: Spatial Velocity Check ────────────────
        g3_pass, disp, scale_chg = gate_velocity(
            self.prev_box, pred_box, self.max_disp, self.max_scale
        )

        # ── Decision ──────────────────────────────────────
        all_pass = g1_pass and g2_pass and g3_pass

        entry = {
            "frame":       self.frame_idx,
            "g1_peak":     (g1_pass, round(norm_peak, 4)),
            "g2_psr":      (g2_pass, round(psr, 4)),
            "g3_vel":      (g3_pass, round(disp, 4), round(scale_chg, 4)),
            "updated":     all_pass,
        }
        self.gate_log.append(entry)

        if all_pass:
            # EMA update: blend current template toward new observation
            # Anchors heavily toward the pristine first frame via (1-α)
            # search_feat may be larger (e.g. 16x16) -> downsample to template size (8x8)
            _, _, Ht, Wt = self.template_feat.shape
            search_resized = F.adaptive_avg_pool2d(search_feat, (Ht, Wt))
            log.debug(f"  [Updater] search_feat {search_feat.shape[-2:]} -> "
                      f"resized to {search_resized.shape[-2:]} for EMA")
            self.template_feat = (
                self.alpha * search_resized +
                (1 - self.alpha) * self.template_feat
            )
            self.prev_box = pred_box.clone()
            self.update_count += 1
            log.debug(f"  [Updater] ✓ UPDATE accepted  "
                      f"(total updates: {self.update_count})")
        else:
            self.reject_count += 1
            reasons = []
            if not g1_pass: reasons.append(f"peak={norm_peak:.3f}<{self.peak_thresh}")
            if not g2_pass: reasons.append(f"PSR={psr:.3f}<{self.psr_thresh}")
            if not g3_pass: reasons.append(f"disp={disp:.3f}>{self.max_disp}")
            log.debug(f"  [Updater] ✗ UPDATE rejected  "
                      f"reason: {', '.join(reasons)}  "
                      f"(total rejects: {self.reject_count})")

        return self.template_feat

    def summary(self) -> dict:
        """Returns a summary of update statistics for this sequence."""
        total = self.update_count + self.reject_count
        accept_rate = self.update_count / total if total > 0 else 0.0
        s = {
            "frames_processed": self.frame_idx,
            "updates_accepted": self.update_count,
            "updates_rejected": self.reject_count,
            "accept_rate":      round(accept_rate, 3),
        }
        log.info(f"[Updater.summary] frames={s['frames_processed']}, "
                 f"accepted={s['updates_accepted']}, "
                 f"rejected={s['updates_rejected']}, "
                 f"accept_rate={s['accept_rate']:.1%}")
        return s


# ──────────────────────────────────────────────────────────────
# SANITY CHECK
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import math

    logging.basicConfig(
        level=logging.DEBUG,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {device}")

    C, Ht, Wt, Hs, Ws = 128, 8, 8, 16, 16

    # ── Init updater ──────────────────────────────────────
    updater = TripleGatedUpdater(
        alpha=0.02,
        peak_thresh=0.5,
        psr_thresh=3.0,
        max_disp=0.25,
        max_scale=0.50,
    )

    first_feat = torch.randn(1, C, Ht, Wt).to(device)
    init_box   = torch.tensor([0.5, 0.5, 0.15, 0.12]).to(device)
    updater.init(first_feat, init_box)

    log.info("\n── Simulating 6 tracking frames ────────────────────")

    scenarios = [
        # (description,             search_feat_type,   pred_box)
        ("Normal frame (good)",     "similar",   [0.51, 0.51, 0.15, 0.12]),
        ("Tiny movement (good)",    "similar",   [0.53, 0.52, 0.15, 0.12]),
        ("Fast motion (gate3 fail)","similar",   [0.80, 0.80, 0.15, 0.12]),
        ("Occlusion (gate1 fail)",  "noise",     [0.53, 0.52, 0.15, 0.12]),
        ("Recovery (good)",         "similar",   [0.54, 0.53, 0.15, 0.12]),
        ("Scale jump (gate3 fail)", "similar",   [0.54, 0.53, 0.40, 0.40]),
    ]

    for desc, feat_type, box_vals in scenarios:
        if feat_type == "similar":
            # Correlated with template → strong response peak
            search_feat = first_feat.repeat(1, 1, Hs // Ht, Ws // Wt)
            search_feat = search_feat + 0.05 * torch.randn_like(search_feat)
        else:
            # Pure noise → no meaningful peak (simulates occlusion)
            search_feat = torch.randn(1, C, Hs, Ws).to(device)

        pred_box = torch.tensor(box_vals).to(device)
        log.info(f"\n  Scenario: {desc}")
        updater.step(search_feat, pred_box)

    log.info("\n── Gate Log ────────────────────────────────────────")
    for entry in updater.gate_log:
        status = "UPDATE ✓" if entry["updated"] else "FROZEN  ✗"
        log.info(
            f"  Frame {entry['frame']:02d} | {status} | "
            f"G1(peak)={entry['g1_peak'][1]:.2f} | "
            f"G2(PSR)={entry['g2_psr'][1]:.2f} | "
            f"G3(disp)={entry['g3_vel'][1]:.3f}"
        )

    log.info("")
    summary = updater.summary()

    # ── Assertions ───────────────────────────────────────
    assert summary["frames_processed"] == 6
    assert summary["updates_accepted"] >= 1,  "At least 1 update should be accepted"
    assert summary["updates_rejected"] >= 2,  "At least 2 updates should be rejected"
    assert updater.first_feat is not None,    "First frame must be preserved"

    # Verify first frame is truly frozen (never modified)
    delta = (updater.first_feat - first_feat).abs().max().item()
    assert delta == 0.0, f"first_feat was modified! delta={delta}"
    log.info("[✓] first_feat is immutable ✓")

    log.info("\n[✓] Triple-Gated Updater sanity check passed.")
    log.info("    Ready for Step 5: Full ZajelTracker integration.")

09:10:29 | INFO     | ZajelUpdater | Device: cuda
09:10:29 | INFO     | ZajelUpdater | [TripleGatedUpdater] α=0.02, peak_thresh=0.5, psr_thresh=3.0, max_disp=0.25, max_scale=0.5
09:10:29 | INFO     | ZajelUpdater | [Updater.init] template_feat=torch.Size([1, 128, 8, 8]), init_box=[0.5, 0.5, 0.15000000596046448, 0.11999999731779099]
09:10:29 | INFO     | ZajelUpdater | 
── Simulating 6 tracking frames ────────────────────
09:10:29 | INFO     | ZajelUpdater | 
  Scenario: Normal frame (good)
09:10:29 | DEBUG    | ZajelUpdater | 
[Updater] Frame 1 ─────────────────────
09:10:29 | DEBUG    | ZajelUpdater |   [XCorr] template=torch.Size([1, 128, 8, 8]), search=torch.Size([1, 128, 16, 16]) → response=torch.Size([1, 1, 9, 9])
09:10:29 | DEBUG    | ZajelUpdater |   [Gate1-Peak] norm_peak=4.3578, threshold=0.50 → ✓ PASS
09:10:29 | DEBUG    | ZajelUpdater |   [Gate2-PSR] peak=7848.8877, sl_mean=363.2979, sl_std=1660.4209, PSR=4.5082, threshold=3.00 → ✓ PASS
09:10:29 | DEBUG    | ZajelUpdater |  

In [11]:
"""
Zajel Team - Full Tracker Integration
Egypt-Japan University of Science and Technology (E-JUST)

Wires together all four components into one end-to-end tracker:

  Step 2: ZajelBackbone        (Mamba SSM dual-stream feature extractor)
  Step 3: ZajelHead            (Classification + NWD Regression)
  Step 4: TripleGatedUpdater   (Online template management)

  ZajelTracker exposes two interfaces:
    .init(frame_bgr, box_xywh)     — initialize on first frame
    .track(frame_bgr)              — track in subsequent frames
    → returns (x, y, w, h) in pixel coordinates

  Training:
    ZajelTracker.forward(template, search, gt_box) → loss dict

  Inference loop is compatible with run_inference_and_save()
  from zajel_dataset_loader.py.
"""

import logging
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Import prior steps (adjust paths for your Kaggle layout) ──
import sys
sys.path.append("/kaggle/working")
from zajel_mamba_backbone import ZajelBackbone
from zajel_heads           import ZajelHead
from zajel_updater         import TripleGatedUpdater

log = logging.getLogger("ZajelTracker")


# ──────────────────────────────────────────────────────────────
# PREPROCESSING HELPERS
# ──────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def crop_and_resize(
    frame_bgr: np.ndarray,
    cx: float, cy: float,
    crop_size: float,
    out_size: int,
) -> np.ndarray:
    """Crop square region centered at (cx,cy), resize to out_size×out_size."""
    H, W = frame_bgr.shape[:2]
    half = crop_size / 2
    x1, y1 = int(cx - half), int(cy - half)
    x2, y2 = int(cx + half), int(cy + half)

    pad_l = max(0, -x1);  pad_t = max(0, -y1)
    pad_r = max(0, x2-W); pad_b = max(0, y2-H)
    if any([pad_l, pad_t, pad_r, pad_b]):
        frame_bgr = cv2.copyMakeBorder(
            frame_bgr, pad_t, pad_b, pad_l, pad_r,
            cv2.BORDER_CONSTANT, value=(114, 114, 114)
        )
    x1+=pad_l; x2+=pad_l; y1+=pad_t; y2+=pad_t
    crop = frame_bgr[y1:y2, x1:x2]
    return cv2.resize(crop, (out_size, out_size))


def to_tensor(patch_bgr: np.ndarray, device: torch.device) -> torch.Tensor:
    """BGR HxWxC uint8 → normalized float tensor (1, 3, H, W)."""
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std  = np.array(IMAGENET_STD,  dtype=np.float32)
    rgb  = (rgb - mean) / std
    t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device)
    return t                                     # (1, 3, H, W)


# ──────────────────────────────────────────────────────────────
# ZAJEL TRACKER
# ──────────────────────────────────────────────────────────────
class ZajelTracker(nn.Module):
    """
    End-to-end aerial object tracker.

    Inference interface (online, frame-by-frame):
        tracker = ZajelTracker().to(device)
        tracker.eval()
        tracker.init(frame0_bgr, (x, y, w, h))
        for frame in video:
            x, y, w, h = tracker.track(frame_bgr)

    Training interface:
        loss_dict = tracker(template_tensor, search_tensor, gt_box)
        loss_dict["loss"].backward()
    """

    TEMPLATE_SIZE = 64
    SEARCH_SIZE   = 128
    SEARCH_FACTOR = 4.0   # search region = SEARCH_FACTOR × max(target_w, target_h)

    def __init__(
        self,
        d_model:     int   = 128,
        depth:       int   = 4,
        patch_size:  int   = 8,
        d_state:     int   = 16,
        expand:      int   = 2,
        head_hidden: int   = 64,
        nwd_C:       float = 0.5,
        ema_alpha:   float = 0.02,
        peak_thresh: float = 0.5,
        psr_thresh:  float = 3.0,
        max_disp:    float = 0.25,
        max_scale:   float = 0.50,
    ):
        super().__init__()
        log.info("=" * 60)
        log.info("[ZajelTracker] Initializing full tracker")

        # ── Step 2: Backbone ──────────────────────────────
        self.backbone = ZajelBackbone(
            d_model=d_model, depth=depth,
            patch_size=patch_size, d_state=d_state, expand=expand,
        )

        # ── Step 3: Heads ─────────────────────────────────
        self.head = ZajelHead(
            d_model=d_model, hidden=head_hidden, nwd_C=nwd_C,
        )

        # ── Step 4: Template Updater (stateful, not nn.Module) ──
        self.updater = TripleGatedUpdater(
            alpha=ema_alpha,
            peak_thresh=peak_thresh,
            psr_thresh=psr_thresh,
            max_disp=max_disp,
            max_scale=max_scale,
        )

        # ── Inference state ───────────────────────────────
        self._device       = torch.device("cpu")
        self._prev_box_px  = None   # (x, y, w, h) in pixel coords
        self._search_scale = None   # pixels-per-unit for current crop
        self._search_cx    = None   # crop center x (pixels)
        self._search_cy    = None   # crop center y (pixels)
        self._crop_size    = None   # size of search crop in pixels

        total = sum(p.numel() for p in self.parameters())
        log.info(f"[ZajelTracker] Total params: {total/1e6:.3f}M / 10M limit")
        log.info("=" * 60)

    # ──────────────────────────────────────────────────────
    # TRAINING FORWARD
    # ──────────────────────────────────────────────────────
    def forward(
        self,
        template: torch.Tensor,    # (B, 3, 64,  64)
        search:   torch.Tensor,    # (B, 3, 128, 128)
        gt_box:   torch.Tensor,    # (B, 4) cx,cy,w,h normalized [0,1]
    ) -> dict:
        """
        Training forward pass.
        Returns loss dict: {loss, cls_loss, reg_loss, pred_box, score_map}
        """
        log.debug(f"[Tracker.forward] template={template.shape}, "
                  f"search={search.shape}, gt_box={gt_box.shape}")

        # ── Feature extraction ────────────────────────────
        feats  = self.backbone(template, search)
        fused  = feats["fused"]                      # (B, d_model, 16, 16)
        log.debug(f"  [Tracker] fused features: {fused.shape}")

        # ── Head + loss ───────────────────────────────────
        out = self.head(fused, gt_box=gt_box)
        log.debug(f"  [Tracker] loss={out['loss'].item():.4f} "
                  f"(cls={out['cls_loss'].item():.4f}, "
                  f"reg={out['reg_loss'].item():.4f})")
        return out

    # ──────────────────────────────────────────────────────
    # INFERENCE — INIT on first frame
    # ──────────────────────────────────────────────────────
    @torch.no_grad()
    def init(self, frame_bgr: np.ndarray, box_xywh: tuple):
        """
        Initialize tracker on the first frame.

        Args:
            frame_bgr: HxWx3 BGR numpy array
            box_xywh:  (x, y, w, h) in pixels — ground truth first-frame box
        """
        x, y, w, h = [float(v) for v in box_xywh]
        cx, cy = x + w / 2, y + h / 2

        log.info(f"[Tracker.init] box=({x:.1f},{y:.1f},{w:.1f},{h:.1f})  "
                 f"center=({cx:.1f},{cy:.1f})")

        # ── Crop template patch ───────────────────────────
        t_crop_sz = max(w, h) * 2.0
        template_patch = crop_and_resize(
            frame_bgr, cx, cy, t_crop_sz, self.TEMPLATE_SIZE
        )
        template_t = to_tensor(template_patch, self._device)  # (1,3,64,64)

        # ── Extract template features ─────────────────────
        template_up = F.interpolate(template_t, size=(128, 128),
                                    mode='bilinear', align_corners=False)
        f_t_full    = self.backbone.backbone(template_up)              # (1,d,16,16)
        f_t         = F.adaptive_avg_pool2d(f_t_full, (8, 8))         # (1,d,8,8)
        f_t_proj    = self.backbone.template_proj(f_t)

        # ── Initialize updater ────────────────────────────
        init_box_norm = torch.tensor([0.5, 0.5, w / (t_crop_sz + 1e-6),
                                      h / (t_crop_sz + 1e-6)],
                                     device=self._device)
        self.updater.init(f_t_proj, init_box_norm)

        # ── Save state for next frame ─────────────────────
        self._prev_box_px  = (x, y, w, h)
        self._search_cx    = cx
        self._search_cy    = cy
        self._crop_size    = max(w, h) * self.SEARCH_FACTOR
        self._template_t   = template_t

        log.info(f"[Tracker.init] Done. Search crop size: {self._crop_size:.1f}px")

    # ──────────────────────────────────────────────────────
    # INFERENCE — TRACK each subsequent frame
    # ──────────────────────────────────────────────────────
    @torch.no_grad()
    def track(self, frame_bgr: np.ndarray) -> tuple:
        """
        Track target in the given frame.

        Returns:
            (x, y, w, h) in pixel coordinates (top-left + size)
        """
        assert self._prev_box_px is not None, "Call .init() before .track()"

        # ── Crop search region around last known position ─
        search_patch = crop_and_resize(
            frame_bgr,
            self._search_cx, self._search_cy,
            self._crop_size, self.SEARCH_SIZE
        )
        search_t = to_tensor(search_patch, self._device)   # (1,3,128,128)

        # ── Extract search features ───────────────────────
        f_s      = self.backbone.backbone(search_t)             # (1,d,16,16)
        f_s_proj = self.backbone.search_proj(f_s)

        # ── Cross-correlate with current template ─────────
        template_feat = self.updater.template_feat              # (1,d,8,8)
        template_up   = F.interpolate(template_feat, size=f_s_proj.shape[-2:],
                                      mode='bilinear', align_corners=False)
        fused = f_s_proj * template_up                          # (1,d,16,16)

        # ── Head prediction ───────────────────────────────
        out      = self.head(fused)
        pred_box = out["pred_box"][0]   # (4,) cx,cy,w,h normalized to search crop

        log.debug(f"  [Tracker.track] pred_box (norm): "
                  f"cx={pred_box[0]:.3f} cy={pred_box[1]:.3f} "
                  f"w={pred_box[2]:.3f} h={pred_box[3]:.3f}")

        # ── Decode to pixel coordinates ───────────────────
        crop_sz = self._crop_size
        cx_abs = self._search_cx - crop_sz/2 + pred_box[0].item() * crop_sz
        cy_abs = self._search_cy - crop_sz/2 + pred_box[1].item() * crop_sz
        w_abs  = pred_box[2].item() * crop_sz
        h_abs  = pred_box[3].item() * crop_sz
        x_abs  = cx_abs - w_abs / 2
        y_abs  = cy_abs - h_abs / 2

        log.debug(f"  [Tracker.track] pred_box (px):   "
                  f"x={x_abs:.1f} y={y_abs:.1f} w={w_abs:.1f} h={h_abs:.1f}")

        # ── Update template (Triple-Gated) ────────────────
        self.updater.step(f_s_proj, pred_box)

        # ── Update search center for next frame ───────────
        self._search_cx = cx_abs
        self._search_cy = cy_abs
        self._prev_box_px = (x_abs, y_abs, w_abs, h_abs)

        return (x_abs, y_abs, w_abs, h_abs)

    def to(self, device):
        self._device = device if isinstance(device, torch.device) else torch.device(device)
        return super().to(device)


# ──────────────────────────────────────────────────────────────
# TRAINING LOOP HELPER
# ──────────────────────────────────────────────────────────────
def train_one_epoch(
    model:      ZajelTracker,
    loader:     torch.utils.data.DataLoader,
    optimizer:  torch.optim.Optimizer,
    device:     torch.device,
    epoch:      int,
    log_every:  int = 50,
) -> dict:
    """
    Runs one training epoch.
    Returns: {"loss": avg, "cls_loss": avg, "reg_loss": avg}
    """
    model.train()
    total_loss = cls_total = reg_total = 0.0
    n_batches  = len(loader)

    log.info(f"[Train] Epoch {epoch} — {n_batches} batches")

    for i, batch in enumerate(loader):
        template = batch["template"].to(device)   # (B, 3, 64,  64)
        search   = batch["search"].to(device)     # (B, 3, 128, 128)
        gt_box   = batch["gt_box"].to(device)     # (B, 4)

        optimizer.zero_grad()
        out = model(template, search, gt_box)
        out["loss"].backward()

        # Gradient clipping — prevents instability early in training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += out["loss"].item()
        cls_total  += out["cls_loss"].item()
        reg_total  += out["reg_loss"].item()

        if (i + 1) % log_every == 0 or (i + 1) == n_batches:
            avg_loss = total_loss / (i + 1)
            avg_cls  = cls_total  / (i + 1)
            avg_reg  = reg_total  / (i + 1)
            log.info(f"  [Epoch {epoch}] batch {i+1:04d}/{n_batches} | "
                     f"loss={avg_loss:.4f}  cls={avg_cls:.4f}  reg={avg_reg:.4f}")

    return {
        "loss":     total_loss / n_batches,
        "cls_loss": cls_total  / n_batches,
        "reg_loss": reg_total  / n_batches,
    }


# ──────────────────────────────────────────────────────────────
# SANITY CHECK
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    logging.basicConfig(
        level=logging.DEBUG,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info(f"Device: {device}")

    # ── Build tracker ─────────────────────────────────────
    tracker = ZajelTracker(
        d_model=128, depth=4, patch_size=8,
        d_state=16,  expand=2,
    ).to(device)
    tracker.eval()

    # ── 1. Training forward pass ──────────────────────────
    log.info("\n── Test 1: Training forward pass ───────────────────")
    B = 4
    template = torch.randn(B, 3, 64,  64 ).to(device)
    search   = torch.randn(B, 3, 128, 128).to(device)
    gt_box   = torch.tensor([
        [0.5, 0.5, 0.15, 0.12],
        [0.3, 0.7, 0.10, 0.08],
        [0.6, 0.4, 0.20, 0.18],
        [0.2, 0.2, 0.12, 0.10],
    ]).to(device)

    tracker.train()
    out = tracker(template, search, gt_box)
    log.info(f"  loss     = {out['loss'].item():.4f}")
    log.info(f"  cls_loss = {out['cls_loss'].item():.4f}")
    log.info(f"  reg_loss = {out['reg_loss'].item():.4f}")
    out["loss"].backward()
    log.info("  Backward pass ✓")

    # ── 2. Inference simulation ───────────────────────────
    log.info("\n── Test 2: Inference (init + 5 track frames) ───────")
    tracker.eval()

    # Simulate a 480×640 frame with a target at (300, 200, 80, 60)
    frame_h, frame_w = 480, 640
    dummy_frame = np.random.randint(0, 255, (frame_h, frame_w, 3), dtype=np.uint8)
    init_box = (300, 200, 80, 60)   # x, y, w, h

    tracker.init(dummy_frame, init_box)
    log.info(f"  Initialized. Search crop: {tracker._crop_size:.1f}px")

    for t in range(5):
        # Simulate slight motion each frame
        noise_frame = dummy_frame.copy()
        pred = tracker.track(noise_frame)
        log.info(f"  Frame {t+1}: pred = "
                 f"x={pred[0]:.1f} y={pred[1]:.1f} "
                 f"w={pred[2]:.1f} h={pred[3]:.1f}")

    updater_summary = tracker.updater.summary()

    # ── 3. Parameter budget ───────────────────────────────
    log.info("\n── Final Parameter Budget ──────────────────────────")
    total = sum(p.numel() for p in tracker.parameters())
    log.info(f"  Total params : {total/1e6:.3f}M  (limit: 10M)")
    log.info(f"  Budget used  : {total/10e6*100:.1f}%")

    # ── Assertions ────────────────────────────────────────
    assert out["loss"].item() > 0,              "Loss must be positive"
    assert out["pred_box"].shape == (B, 4),     "pred_box shape error"
    assert total < 10_000_000,                  f"Exceeds 10M params! ({total/1e6:.2f}M)"
    assert isinstance(pred, tuple) and len(pred) == 4, "track() must return (x,y,w,h)"

    log.info("\n[✓] Full ZajelTracker integration check passed.")
    log.info("    Ready for Step 6: Training loop + submission.")

ModuleNotFoundError: No module named 'zajel_mamba_backbone'